In [1]:
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(parent_dir)
task_name = 'MAK_3s_5r_NLP_MAK_REINFORCE_SIL_COUNCIL'
print('Working directory set to:', parent_dir)

Working directory set to: /local0/rossin/git/CRN-GenerativeAI


### Imports

In [2]:
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter
from datetime import datetime
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
import numpy as np
from itertools import product
from tqdm import tqdm
import time

from RL4CRN.environments.environment import Environment
from RL4CRN.environments.parallel_environments import ParallelEnvironments
from RL4CRN.environments.serial_environments import SerialEnvironments
from RL4CRN.agents.reinforce_agent import REINFORCEAgent
from RL4CRN.policies.add_reaction_by_ordered_index import AddReactionByOrderedIndex
from RL4CRN.policies.add_reaction_by_index import AddReactionByIndex

# Import Interface packages
from RL4CRN.env2agent_interface.explicit_observer import ExplicitObserver
from RL4CRN.env2agent_interface.explicit_tensorizer import ExplicitTensorizer
from RL4CRN.agent2env_interface.library_actuator import LibraryActuator
from RL4CRN.agent2env_interface.iocrn_stepper import IOCRNStepper

# Import CRN packages
from RL4CRN.iocrns.iocrn import IOCRN
from RL4CRN.iocrns.reactions import MassAction
from RL4CRN.utils.ic import IC
from RL4CRN.iocrns.reaction_library import construct_mass_action_library

# Import Reward packages
from RL4CRN.rewards.deterministic import dynamic_tracking_error

# --- NEW IMPORT ---
# Replacing VertexMultiAgentDebate with your new Council class
from RL4CRN.NLPAgent.Councils.LinearCRNCouncil import LinearCRNCouncil

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/local0/home/rossin/.keys/crn-evolution-be2b980ea837.json"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
api_key = "vhIR3uyqsKyU4L7SA8fLCfTSC"
logger = CometLogger(
    api_key=api_key,
    project=task_name,        
    workspace="redsnic", 
    name=f'{task_name}_{timestamp}',
)
logger = logger.experiment

/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/redsnic/mak-3s-5r-nlp-mak-reinforce-sil-council/c852fd1b664f46fd91e1491bf98d1f16



### Template CRN

In [3]:
# Construct the template CRN
scale = 1.0
r1 = MassAction(reactant_labels=[], product_labels=['Z_1'], input_channels=['u_1'], params=[scale], params_controllability=[True])
r2 = MassAction(reactant_labels=['X_1'], product_labels=[], input_channels=['u_2'], params=[1.], params_controllability=[True])
crn_template = IOCRN([r1, r2], output_labels=['X_1'])
crn_template.compile()
p = crn_template.num_inputs 
print("Template CRN:")
print(crn_template)

# Construct the library of possible reactions
species_labels = ['X_1', 'Z_1', 'Z_2']
library = construct_mass_action_library(species_labels=species_labels, order=2)
crn_template.set_library_context(library)
M = len(library.reactions) 
K = library.get_num_parameters() 
print("Library of possible reactions:")
print(library)
print("------------------------------------------------")

Template CRN:
Inputs: ['u_1', 'u_2'] 
Species: ['X_1', 'Z_1'] 
Output Species: ['X_1'] 
∅ ----> Z_1;  [MAK(1.0, u_1)]
X_1 ----> ∅;  [MAK(1.0, u_2)]
Library of possible reactions:
Number of reactions: 91
R0: ∅ ----> ∅;  [MAK(None)]
R1: ∅ ----> X_1;  [MAK(None)]
R2: ∅ ----> Z_1;  [MAK(None)]
R3: ∅ ----> Z_2;  [MAK(None)]
R4: ∅ ----> X_1 + X_1;  [MAK(None)]
R5: ∅ ----> X_1 + Z_1;  [MAK(None)]
R6: ∅ ----> X_1 + Z_2;  [MAK(None)]
R7: ∅ ----> Z_1 + Z_1;  [MAK(None)]
R8: ∅ ----> Z_1 + Z_2;  [MAK(None)]
R9: ∅ ----> Z_2 + Z_2;  [MAK(None)]
R10: X_1 ----> ∅;  [MAK(None)]
R11: X_1 ----> Z_1;  [MAK(None)]
R12: X_1 ----> Z_2;  [MAK(None)]
R13: X_1 ----> X_1 + X_1;  [MAK(None)]
R14: X_1 ----> X_1 + Z_1;  [MAK(None)]
R15: X_1 ----> X_1 + Z_2;  [MAK(None)]
R16: X_1 ----> Z_1 + Z_1;  [MAK(None)]
R17: X_1 ----> Z_1 + Z_2;  [MAK(None)]
R18: X_1 ----> Z_2 + Z_2;  [MAK(None)]
R19: Z_1 ----> ∅;  [MAK(None)]
R20: Z_1 ----> X_1;  [MAK(None)]
R21: Z_1 ----> Z_2;  [MAK(None)]
R22: Z_1 ----> X_1 + X_1;  [MAK(Non

### Setup

In [4]:
# Device Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'Number of CPUs available: {os.cpu_count()}')


# Flags and filenames
save_flag = True                                                
load_flag = False                                                
train_flag = True                                               
save_sheet_flag = True                                          

save_filename = timestamp + '.pth'                              
load_filename = ''                                              
file_name = f"{task_name}.xlsx"

Using device: cuda
Number of CPUs available: 128


In [5]:
# Hyperparameters
max_added_reactions = 5                             
N_CPUs = os.cpu_count()                             
N = 10*N_CPUs                                       
width = 1024                                        
depth = 5                                           
deep_layer_size = 1024*10                           
learning_rate = 1e-4                                
hall_of_fame_size = 100                              
entropy_scheduler = {                               
    'entropy_weight': 1e-3, 
    'topk_entropy_weight' : 1.0,
    'remainder_entropy_weight' : 1.0,
    'entropy_update_coefficient': 1, 
    'entropy_schedule': 1000, 
    'minimum_entropy_weight': 0.0
}
entropy_weights_per_head = {'structure': 2.0, 'continuous': 1.0, 'discrete': 0.0, 'input_influence': 0.0} 
structure_head_temperature = {"target_entropy_ratio_to_max": np.log(5)/np.log(M), "initial_temperature": 1.0, "rate": 0.0, "current_temperature": 1.0}
risk_scheduler = {                                  
    'risk': 0.9, 
    'risk_update': 0.0, 
    'max_risk': 1.0, 
    'risk_schedule': 1000
}
epoch_num = 300                                     
render_schedule = 10                                 
render_mode = {                                     
    'style': 'logger', 
    'task': 'transients', 
    'format': 'image',
    'topology': True,
    'bounds': [2.5]
}
ordering_parameters = {
    'enforce_ordering': False,
    'constraint_weight' : float('inf')
}
sil_settings = {
    'sil_loss_weight': 1.0,
    'sil_use_adaptive_baseline': False,
    'sil_baseline_annealing_rate': 0.95
}
render_n_best = 10                                                    
render_disregard_percentage = 0.99                                    
continuous_distribution = {"type": 'lognormal_1D'}

# Simulation Params
t_f = 100                                           
N_t = 1000                                          
time_horizon = np.linspace(0, t_f, N_t, dtype=np.float32)

# Inputs/Setpoints
nums = [0.5, 1.0, 1.5]
u_list = [np.array(u) for u in product(nums, repeat=p)]
r_list = [np.array([u[0]]) for u in u_list]
ic = IC(names=species_labels, values=[[0.0, 0.0, 0.0, 0.0]])

w = np.ones(N_t)
w[(len(w)//5)*4:] = w[(len(w)//5)*4:]*2
w[:(len(w)//5)] = w[:(len(w)//5)]*0.25
w = w[np.newaxis, :]

def compute_reward(state):
    x0_list = ic.get_ic(state)
    return dynamic_tracking_error(state, u_list, x0_list, time_horizon, r_list, w, norm=1, LARGE_NUMBER=1e4)

### Save to Excell

In [6]:
if save_sheet_flag:
    sheet_name = "Data"
    headers = [
        "Timestamp", "URL",
        "Epochs Completed", "Successful", "Saved", "Comments",
        "Learning Rate", "Epochs #",
        "(m, n, p, N)",
        "NN Depth", "NN Width", "Deep Layer Size", "CPUs #",
        "Entropy Scheduler",
        "Risk Scheduler",
        "Render Schedule", "HoF Size",
        "Simulation Time", "Time Steps #",
        "Initial Conditions #", "Input Scenarios#",
        "Continuous Distribution", "Entropy Weights per Head",
        "Structure Head Temperature",
        "Ordering Enforced",
        "SIL Settings"
    ]

    data_row = [
        timestamp, logger.url,
        None, None, None, None,
        learning_rate, epoch_num,
        str((max_added_reactions, len(species_labels), p, N)),
        depth, width, deep_layer_size, N_CPUs,
        str(entropy_scheduler),
        str(risk_scheduler),
        render_schedule, hall_of_fame_size,
        t_f, N_t, len(ic.values), len(u_list),
        str(continuous_distribution), str(entropy_weights_per_head),
        str(structure_head_temperature),
        f"Yes: {ordering_parameters['constraint_weight']}" if ordering_parameters['enforce_ordering'] else "No",
        str(sil_settings)
    ]

    if os.path.exists(file_name):
        wb = load_workbook(file_name)
        if sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
        else:
            ws = wb.create_sheet(sheet_name)
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = sheet_name

    # Write headers if sheet is empty
    if ws.max_row == 1 and ws.max_column == 1 and ws.cell(row=1, column=1).value is None:
        for col, header in enumerate(headers, start=1):
            ws.cell(row=1, column=col, value=header)

    # Append experiment as next row
    next_row = ws.max_row + 1
    for col, value in enumerate(data_row, start=1):
        ws.cell(row=next_row, column=col, value=value)

    # Freeze header row and add filter
    ws.freeze_panes = "B1" 
    ws.auto_filter.ref = ws.dimensions

    # === Auto-fit column widths (except URL column) ===
    # URL column is column 2 (B), we leave its width unchanged.
    url_col_index = 2

    for col in range(1, ws.max_column + 1):
        if col == url_col_index:
            continue  # keep URL column width as-is

        max_length = 0
        for row in range(1, ws.max_row + 1):
            cell = ws.cell(row=row, column=col)
            value = cell.value
            if value is not None:
                # Convert to string to measure length
                length = len(str(value))
                if length > max_length:
                    max_length = length

        # Some padding so text isn't touching the cell border
        adjusted_width = max_length + 2 if max_length > 0 else 10
        col_letter = get_column_letter(col)
        ws.column_dimensions[col_letter].width = adjusted_width

    wb.save(file_name)
    print(f"New experiment data saved in row {next_row} of '{file_name}'.")

New experiment data saved in row 11 of 'MAK_3s_5r_NLP_MAK_REINFORCE_SIL_COUNCIL.xlsx'.


### Model setup

In [7]:
# Construct parallel environments
crn_0 = crn_template.clone()
mult_env = ParallelEnvironments([Environment(crn_0, max_added_reactions, logger=logger, logger_schedule=1) for _ in range(N)], hall_of_fame_size=hall_of_fame_size, N_CPUs=N_CPUs, logger=logger)

# Construct the policy
encoder_attributes = {"hidden_size": width, "num_layers": depth}
structure_head_attributes = {"hidden_size": width, "num_layers": depth}
rate_head_attributes = {"hidden_size": width, "num_layers": depth}
input_influence_head_attributes = {"hidden_size": width, "num_layers": depth}
masks = {"continuous": library.get_parameter_mask(mode="continuous"), "discrete": library.get_parameter_mask(mode="discrete"), "logit": library.get_logit_mask()}

if ordering_parameters["enforce_ordering"]:
    policy = AddReactionByOrderedIndex(M, K, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, target_set_size=crn_template.num_reactions+max_added_reactions, allow_input_influence=False, masks=masks, device=device, continuous_distribution=continuous_distribution, entropy_weights_per_head=entropy_weights_per_head, combinatorial_bias_enabled=ordering_parameters["enforce_ordering"], constraint_strength=ordering_parameters["constraint_weight"])
else:
    policy = AddReactionByIndex(M, K, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, allow_input_influence=False, masks=masks, device=device, continuous_distribution=continuous_distribution, entropy_weights_per_head=entropy_weights_per_head)

# Construct the agent
agent = REINFORCEAgent(policy, allow_input_influence=False, logger=logger, learning_rate=learning_rate, entropy_scheduler=entropy_scheduler, risk_scheduler=risk_scheduler, sil_settings=sil_settings, device=device)
if load_flag:
    agent.policy.load_state_dict(torch.load(load_filename+'.pth', map_location=device))

# Construct the interfaces
observer = ExplicitObserver(reaction_library=library, allow_input_observation=False)
tensorizer = ExplicitTensorizer(device=device)
actuator = LibraryActuator(reaction_library=library)
stepper = IOCRNStepper()

### Training loop

In [8]:
# ==========================================
# 0. SETUP & UTILS
# ==========================================
IS_ORDERED_POLICY = "Ordered" in agent.policy.__class__.__name__
print(f"Policy detected: {agent.policy.__class__.__name__}")
print(f"Gemini Injection Mode: {'SORTED (Ordered Trajectory)' if IS_ORDERED_POLICY else 'UNSORTED (Permutation Invariant)'}")

# ==========================================
# 1. LINEAR CRN COUNCIL CONFIGURATION
# ==========================================
PROJECT_ID = "crn-evolution"

gemini_task_desc = (
    f"Implement a Chemical Reaction Network that achieves Robust Perfect Adaptation (RPA) via Integral Control. "
    f"The system has 2 inputs: u1 (Reference Setpoint) and u2 (Disturbance). "
    f"Goal 1 (Tracking): The output species 'r' must converge exactly to the concentration of u1 at steady state. "
    f"Goal 2 (Robustness): The output must remain at u1 regardless of the value of u2 (the disturbance). "
    f"Select exactly {max_added_reactions} reactions."
)

# Instantiate the Council
council_system = LinearCRNCouncil(
    project_id=PROJECT_ID, 
    location="global"
)

# --- DEFINE SPLIT CONTEXTS (The "Board") ---
# 1. Conceptual Library (For reasoning agents: Narrator -> Player)
library_description = (
    f"Mass Action Kinetics (Order 2).\n"
    f"Available Species: {species_labels}\n"
    f"Valid Reaction Types:\n"
    f" - Unimolecular: A -> B (or A -> B + C)\n"
    f" - Bimolecular:  A + B -> C (or A + B -> C + D)\n"
    f" - Synthesis:    0 -> A\n"
    f" - Degradation:  A -> 0\n"
    f"*Do not concern yourself with reaction indices. Focus on topology.*"
)

# 2. Explicit Library (For the Writer ONLY)
library_explicit_str = str(library)

gemini_schedule = 10  

# ==========================================
# 2. TRAINING LOOP
# ==========================================
if train_flag:
    agent.policy.train()
    warm_start_triggered = False
    debate_transcript_file = f"council_transcript_{task_name}_{time.strftime('%Y%m%d_%H%M%S')}.txt"

    for i in tqdm(range(epoch_num)):
        
        # --- A. Standard RL Step ---
        mult_env.reset()
        for j in range(max_added_reactions):
            observations = mult_env.observe(observer, tensorizer)
            actions, raw_actions = agent.act(observations, actuator)
            out = mult_env.step(actions, stepper, raw_actions=raw_actions)
        
        rewards = mult_env.get_reward(compute_reward)
        mult_env.hall_of_fame.add_all(mult_env.envs)

        successful_count = sum(1 for env in mult_env.envs if not env.state.last_task_info.get('has_diverged', False))
        if logger:
            logger.log_metric("Successful Environments (%)", successful_count/N, step=i)
        
        # --- B. GEMINI COUNCIL PHASE ---
        should_run_debate = (i > 0 and i % gemini_schedule == 0) or (i == 1 and not warm_start_triggered)

        if should_run_debate:
            if i == 1: warm_start_triggered = True
            
            start_time = time.time()
            print(f"\n[Gemini] Epoch {i}: Convening the Council...")
            
            # 1. Run the Council Session 

            candidates, transcript = council_system.run_debate_session(
                task_desc=gemini_task_desc,
                hof_iter=mult_env.hall_of_fame,
                library_description=library_description,     # <--- Conceptual Context
                library_explicit_str=library_explicit_str,   # <--- Syntax Context (Writer only)
                max_added_reactions=max_added_reactions
            )
            
            # 2. Save Transcript
            with open(debate_transcript_file, "a", encoding="utf-8") as f:
                f.write(f"\n\n=== EPOCH {i} ===\n")
                f.write(transcript)
            print(f"[Gemini] Transcript appended to {debate_transcript_file}.")
            
            # 3. Evaluate, Transplant & Track
            new_gemini_envs = council_system.evaluate_and_transplant(
                candidates=candidates,
                crn_template=crn_template,
                max_added_reactions=max_added_reactions,
                library=library,
                stepper=stepper,
                actuator=actuator,
                compute_reward_func=compute_reward,
                is_ordered_policy=IS_ORDERED_POLICY,
                logger=logger
            )

            # 4. Inject into Hall of Fame
            if new_gemini_envs:
                mult_env.hall_of_fame.add_all(new_gemini_envs)

            elapsed_time = time.time() - start_time
            print(f"[Gemini] Council Adjourned in {elapsed_time:.2f}s. {len(new_gemini_envs)} candidates ratified.")
            
            if logger:
                logger.log_metric("Gemini Candidates", len(new_gemini_envs), step=i)
                logger.log_metric("Gemini Duration (s)", elapsed_time, step=i)
                if len(mult_env.hall_of_fame) > 0:
                    best_env = mult_env.hall_of_fame[0] 
                    logger.log_metric("HoF Best Loss", best_env.state.last_task_info.get('reward'), step=i)

        # --- C. Agent Update ---
        agent.update(
            rewards, 
            step_iteration=i, 
            hof=mult_env.hall_of_fame, 
            observer=observer, 
            tensorizer=tensorizer, 
            stepper=stepper, 
            use_sil=True, 
            sil_weighting_scheme='uniform', 
            sil_batch_size=None
        )

        if i % render_schedule == 0:
            mult_env.render(rewards, n_best=render_n_best, disregarded_percentage=render_disregard_percentage, mode=render_mode)

/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Policy detected: AddReactionByIndex
Gemini Injection Mode: UNSORTED (Permutation Invariant)


  0%|          | 1/300 [00:31<2:34:32, 31.01s/it]


[Gemini] Epoch 1: Convening the Council...
--- [DebateGraph] Starting Epoch ---
 -> Narrator is thinking...
 -> Opportunist is thinking...
 -> Contrarian is thinking...
 -> Skeptic is thinking...
 -> Player is thinking...
 -> Writer is thinking...
    (Using specialized context for Writer)
[Gemini] Transcript appended to council_transcript_MAK_3s_5r_NLP_MAK_REINFORCE_SIL_COUNCIL_20251218_115629.txt.
[Debate] Simulating 7 candidates...
  -> Valid. Loss: 0.003345
  -> Valid. Loss: 0.004041
  -> Valid. Loss: 0.003288
  -> Valid. Loss: 0.003888
  -> Valid. Loss: 0.003177
  -> Valid. Loss: 0.002711
  -> Valid. Loss: 0.012287
[Gemini] Council Adjourned in 539.81s. 7 candidates ratified.


  3%|▎         | 10/300 [12:26<2:20:37, 29.09s/it] 

...

 90%|█████████ | 271/300 [4:07:17<56:26, 116.78s/it]/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 93%|█████████▎| 280/300 [4:10:25<08:20, 25.04s/it] 


[Gemini] Epoch 280: Convening the Council...
--- [DebateGraph] Starting Epoch ---
 -> Narrator is thinking...
 -> Opportunist is thinking...
 -> Contrarian is thinking...
 -> Skeptic is thinking...
 -> Player is thinking...
 -> Writer is thinking...
    (Using specialized context for Writer)
[Gemini] Transcript appended to council_transcript_MAK_3s_5r_NLP_MAK_REINFORCE_SIL_COUNCIL_20251218_115629.txt.
[Debate] Simulating 1 candidates...
  -> Valid. Loss: 0.000253
[Gemini] Council Adjourned in 341.74s. 1 candidates ratified.


 97%|█████████▋| 290/300 [4:19:38<04:11, 25.13s/it] 


[Gemini] Epoch 290: Convening the Council...
--- [DebateGraph] Starting Epoch ---
 -> Narrator is thinking...
 -> Opportunist is thinking...
 -> Contrarian is thinking...
 -> Skeptic is thinking...
 -> Player is thinking...
 -> Writer is thinking...
    (Using specialized context for Writer)
[Gemini] Transcript appended to council_transcript_MAK_3s_5r_NLP_MAK_REINFORCE_SIL_COUNCIL_20251218_115629.txt.
[Debate] Simulating 10 candidates...
  -> Valid. Loss: 0.000254
  -> Valid. Loss: 0.000251
  -> Valid. Loss: 0.000254
  -> Valid. Loss: 0.000252
  -> Valid. Loss: 0.000269
  -> Valid. Loss: 0.000252
  -> Valid. Loss: 0.000267
  -> Valid. Loss: 0.000253
  -> Valid. Loss: 0.000254
  -> Valid. Loss: 0.000256
[Gemini] Council Adjourned in 439.99s. 10 candidates ratified.


100%|██████████| 300/300 [4:30:27<00:00, 54.09s/it] 
